Pertemuan 7 — Evaluasi Model: Kode Lengkap Slide & Lab (solusi dosen)

In [4]:
from google.colab import files
import io, pandas as pd

unggah = files.upload()                      # pilih Drive
nama = next(iter(unggah))
d = pd.read_csv(io.BytesIO(unggah[nama]))
print(nama, d.shape)                         # harus (119390, 32)
d = d[d["hotel"] == "Resort Hotel"].reset_index(drop=True)

Saving hotel_bookings.csv to hotel_bookings (1).csv
hotel_bookings (1).csv (119390, 32)


KODE #1

In [7]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

d = pd.read_csv("hotel_bookings.csv")               # 119.390 booking, 32 kolom
fitur = ["hotel", "lead_time", "arrival_date_month", "stays_in_weekend_nights",
         "stays_in_week_nights", "adults", "market_segment", "is_repeated_guest",
         "previous_cancellations", "booking_changes", "deposit_type", "customer_type",
         "days_in_waiting_list", "adr", "required_car_parking_spaces",
         "total_of_special_requests", "country"]
X = d[fitur].copy()
y = d["is_canceled"]                                 # 1 = batal, 0 = datang
X["country"] = X["country"].where(X["country"].isin(X["country"].value_counts().index[:10]), "Lainnya")
X = pd.get_dummies(X, drop_first=True)               # teks -> kolom 0/1

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=77)
print("latihan:", len(Xtr), " ujian:", len(Xte), " batal:", ytr.mean().round(4), yte.mean().round(4))

pemalas = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
print("pemalas (tebak 'datang' semua), akurasi ujian:", round(pemalas.score(Xte, yte), 4))
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=77).fit(Xtr, ytr)
print("hutan  akurasi latihan:", round(rf.score(Xtr, ytr), 4), " ujian:", round(rf.score(Xte, yte), 4))

latihan: 95512  ujian: 23878  batal: 0.3704 0.3704
pemalas (tebak 'datang' semua), akurasi ujian: 0.6296
hutan  akurasi latihan: 0.9392  ujian: 0.8751


Slide 4: menghafal vs paham (pohon makin dalam)

In [8]:
from sklearn.tree import DecisionTreeClassifier
for dalam in [3, 8, 20, None]:
    pohon = DecisionTreeClassifier(max_depth=dalam, random_state=77).fit(Xtr, ytr)
    print("kedalaman", dalam, "latihan:", round(pohon.score(Xtr, ytr), 4),
          "ujian:", round(pohon.score(Xte, yte), 4))

kedalaman 3 latihan: 0.758 ujian: 0.7554
kedalaman 8 latihan: 0.8251 ujian: 0.8222
kedalaman 20 latihan: 0.9149 ujian: 0.8418
kedalaman None latihan: 0.995 ujian: 0.8365


Slide 6: kebocoran soal — kolom yang sudah tahu jawabannya


In [9]:
bocor = X.copy()
# kolom ini baru ada SETELAH tamu menginap atau batal
bocor["status_akhir"] = (d["reservation_status"] != "Check-Out").astype(int)
a, b, c, e = train_test_split(bocor, y, test_size=0.2, stratify=y, random_state=77)
pohon = DecisionTreeClassifier(max_depth=3, random_state=77).fit(a, c)
print("akurasi pohon dengan kolom bocor:", pohon.score(b, e))
print(d.groupby("reservation_status")["is_canceled"].agg(["mean", "size"]))

akurasi pohon dengan kolom bocor: 1.0
                    mean   size
reservation_status             
Canceled             1.0  43017
Check-Out            0.0  75166
No-Show              1.0   1207


Kode #2 — Slide 11: confusion matrix, ketepatan, jangkauan, F1

In [10]:
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

tebak = rf.predict(Xte)
dibiarkan, salah_tuduh, lolos, tertangkap = confusion_matrix(yte, tebak).ravel()
print("dibiarkan", dibiarkan, "| salah tuduh", salah_tuduh, "| lolos", lolos, "| tertangkap", tertangkap)
print(classification_report(yte, tebak, target_names=["datang", "batal"], digits=3))

logistik = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
ukuran = (accuracy_score, precision_score, recall_score, f1_score)
for nama, m in [("logistik", logistik), ("hutan", rf)]:
    t = m.predict(Xte)
    print(nama, "akurasi/ketepatan/jangkauan/F1:", [round(f(yte, t), 3) for f in ukuran])

dibiarkan 14073 | salah tuduh 960 | lolos 2023 | tertangkap 6822
              precision    recall  f1-score   support

      datang      0.874     0.936     0.904     15033
       batal      0.877     0.771     0.821      8845

    accuracy                          0.875     23878
   macro avg      0.875     0.854     0.862     23878
weighted avg      0.875     0.875     0.873     23878

logistik akurasi/ketepatan/jangkauan/F1: [0.805, 0.799, 0.631, 0.705]
hutan akurasi/ketepatan/jangkauan/F1: [0.875, 0.877, 0.771, 0.821]


Kode #3 — Slide 16: batas curiga, ROC/PR-AUC, dan biaya rupiah (hotel butik)

In [11]:
from sklearn.metrics import roc_auc_score, average_precision_score
p = rf.predict_proba(Xte)[:, 1]                     # peluang batal tiap booking (0-1)
print("ROC-AUC:", round(roc_auc_score(yte, p), 3), " PR-AUC:", round(average_precision_score(yte, p), 3))

BIAYA_SALAH_TUDUH = 2_500_000   # hotel butik: tamu datang, kamar sudah dijual ulang -> relokasi (asumsi)
BIAYA_LOLOS       = 1_000_000   # kamar kosong semalam karena batal tak terdeteksi (asumsi)
for batas in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    curiga = p >= batas
    sf = (curiga & (yte == 0)).sum()                # salah tuduh
    lo = (~curiga & (yte == 1)).sum()               # lolos
    rupiah = (sf * BIAYA_SALAH_TUDUH + lo * BIAYA_LOLOS) / len(yte) * 1000 / 1e6
    print(batas, "ketepatan", round(precision_score(yte, curiga), 3),
          "jangkauan", round(recall_score(yte, curiga), 3),
          "biaya Rp", round(rupiah, 1), "juta/1.000 booking")

ROC-AUC: 0.946  PR-AUC: 0.924
0.2 ketepatan 0.654 jangkauan 0.953 biaya Rp 483.6 juta/1.000 booking
0.3 ketepatan 0.749 jangkauan 0.899 biaya Rp 317.0 juta/1.000 booking
0.4 ketepatan 0.818 jangkauan 0.84 biaya Rp 232.6 juta/1.000 booking
0.5 ketepatan 0.877 jangkauan 0.771 biaya Rp 185.2 juta/1.000 booking
0.6 ketepatan 0.919 jangkauan 0.69 biaya Rp 171.0 juta/1.000 booking
0.7 ketepatan 0.954 jangkauan 0.601 biaya Rp 174.9 juta/1.000 booking


Kode #4 — Slide 18: uji berputar 5 lipatan (cross-validation)

In [12]:
from sklearn.model_selection import StratifiedKFold, cross_validate
lipat = StratifiedKFold(n_splits=5, shuffle=True, random_state=77)
model = RandomForestClassifier(n_estimators=100, min_samples_leaf=2, n_jobs=-1, random_state=77)
uji = cross_validate(model, Xtr, ytr, cv=lipat, scoring=["accuracy", "recall", "f1"])
for k in ["test_accuracy", "test_recall", "test_f1"]:
    print(k, uji[k].round(4), "rata-rata", uji[k].mean().round(4), "+/-", uji[k].std().round(4))

test_accuracy [0.8687 0.8719 0.8668 0.8715 0.8702] rata-rata 0.8698 +/- 0.0019
test_recall [0.766  0.7702 0.7668 0.772  0.7709] rata-rata 0.7692 +/- 0.0024
test_f1 [0.8121 0.8166 0.81   0.8165 0.8148] rata-rata 0.814 +/- 0.0026


Slide 17: satu kali ujian bisa kebetulan (hotel kecil, 300 booking)

In [13]:
kecil = X.sample(300, random_state=1)               # hotel kecil: hanya 300 booking
y_kecil = y.loc[kecil.index]
akurasi = []
for s in range(10):                                 # 10 pembagian acak yang berbeda
    a, b, c, e = train_test_split(kecil, y_kecil, test_size=0.2, stratify=y_kecil, random_state=s)
    akurasi.append(DecisionTreeClassifier(max_depth=5, random_state=77).fit(a, c).score(b, e))
print("10 pembagian:", np.round(akurasi, 3))
print("terendah", round(min(akurasi), 3), " tertinggi", round(max(akurasi), 3))
from sklearn.model_selection import cross_val_score
lipat5 = StratifiedKFold(5, shuffle=True, random_state=77)
cv5 = cross_val_score(DecisionTreeClassifier(max_depth=5, random_state=77), kecil, y_kecil, cv=lipat5)
print("uji berputar 5 kali:", cv5.round(3), " rata-rata", cv5.mean().round(3))

10 pembagian: [0.817 0.817 0.783 0.817 0.883 0.733 0.85  0.833 0.733 0.717]
terendah 0.717  tertinggi 0.883
uji berputar 5 kali: [0.733 0.817 0.8   0.733 0.783]  rata-rata 0.773


Slide 19: menghafal vs meremehkan — kedalaman pohon, latihan vs ujian-berputar

In [14]:
lipat = StratifiedKFold(n_splits=5, shuffle=True, random_state=77)
for dalam in [1, 3, 5, 8, 12, 20, None]:
    pohon = DecisionTreeClassifier(max_depth=dalam, random_state=77)
    r = cross_validate(pohon, Xtr, ytr, cv=lipat, return_train_score=True, scoring="accuracy")
    print("kedalaman", dalam, "latihan", r["train_score"].mean().round(3),
          "ujian-berputar", r["test_score"].mean().round(3))

kedalaman 1 latihan 0.751 ujian-berputar 0.751
kedalaman 3 latihan 0.759 ujian-berputar 0.759
kedalaman 5 latihan 0.799 ujian-berputar 0.798
kedalaman 8 latihan 0.826 ujian-berputar 0.824
kedalaman 12 latihan 0.85 ujian-berputar 0.838
kedalaman 20 latihan 0.92 ujian-berputar 0.842
kedalaman None latihan 0.995 ujian-berputar 0.834


Kode #5 — Slide 20: metrik regresi (harga kamar/ADR)

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
Xr, yr = X.drop(columns=["adr"]), d["adr"]          # tebak harga kamar per malam (ADR)
a, b, c, e = train_test_split(Xr, yr, test_size=0.2, random_state=77)
harga = LinearRegression().fit(a, c).predict(b)
rata2 = np.full(len(e), c.mean())                   # tebakan pemalas: selalu harga rata-rata
for nama, t in [("model", harga), ("pemalas", rata2)]:
    print(nama, "MAE", round(mean_absolute_error(e, t), 2),
          "RMSE", round(mean_squared_error(e, t) ** 0.5, 2), "R2", round(r2_score(e, t), 3))

model MAE 27.2 RMSE 38.03 R2 0.388
pemalas MAE 36.48 RMSE 48.61 R2 -0.0


Slide 14: data timpang (simulasi hotel dengan ~5% booking batal)

In [16]:
# Data timpang: hotel dengan ~5% booking batal (SIMULASI: sebagian booking batal dibuang)
neg = X[y == 0]
pos = X[y == 1].sample(int(len(neg) * 0.05 / 0.95), random_state=77)
Xt = pd.concat([neg, pos])
yt = pd.Series([0] * len(neg) + [1] * len(pos), index=Xt.index)
a, b, c, e = train_test_split(Xt, yt, test_size=0.2, stratify=yt, random_state=77)
print("tingkat batal:", yt.mean().round(3), " jumlah booking:", len(yt))
hutan = dict(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=77)
for nama, m in [("pemalas", DummyClassifier(strategy="most_frequent")),
                ("hutan biasa", RandomForestClassifier(**hutan)),
                ("hutan seimbang", RandomForestClassifier(class_weight="balanced_subsample", **hutan))]:
    t = m.fit(a, c).predict(b)
    print(nama, "akurasi", round(accuracy_score(e, t), 3),
          "ketepatan", round(precision_score(e, t, zero_division=0), 3),
          "jangkauan", round(recall_score(e, t), 3))

tingkat batal: 0.05  jumlah booking: 79122
pemalas akurasi 0.95 ketepatan 0.0 jangkauan 0.0
hutan biasa akurasi 0.969 ketepatan 1.0 jangkauan 0.386
hutan seimbang akurasi 0.964 ketepatan 0.685 jangkauan 0.511


Slide 21: jangkauan per segmen pasar

In [17]:
seg = pd.DataFrame({"segmen": d.loc[Xte.index, "market_segment"].values,
                    "y": yte.values, "t": rf.predict(Xte)})
for s, g in seg.groupby("segmen"):
    if len(g) > 500:
        print(f"{s:14s} booking {len(g):5d}  batal asli {g.y.mean():.3f}  "
              f"jangkauan {recall_score(g.y, g.t):.3f}  ketepatan {precision_score(g.y, g.t):.3f}")

Corporate      booking  1094  batal asli 0.191  jangkauan 0.622  ketepatan 0.985
Direct         booking  2448  batal asli 0.150  jangkauan 0.279  ketepatan 0.864
Groups         booking  3878  batal asli 0.610  jangkauan 0.978  ketepatan 0.981
Offline TA/TO  booking  4864  batal asli 0.341  jangkauan 0.898  ketepatan 0.948
Online TA      booking 11395  batal asli 0.370  jangkauan 0.661  ketepatan 0.773


LAB KELAS (Resort Hotel) — SOLUSI DOSEN

In [18]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score

d = pd.read_csv("hotel_bookings.csv")
d = d[d["hotel"] == "Resort Hotel"].reset_index(drop=True)     # lab ini: Resort Hotel (40.060 booking)
fitur = ["lead_time", "arrival_date_month", "stays_in_weekend_nights", "stays_in_week_nights",
         "adults", "market_segment", "is_repeated_guest", "previous_cancellations",
         "booking_changes", "deposit_type", "customer_type", "days_in_waiting_list", "adr",
         "required_car_parking_spaces", "total_of_special_requests", "country"]
X = d[fitur].copy()
y = d["is_canceled"]
X["country"] = X["country"].where(X["country"].isin(X["country"].value_counts().index[:10]), "Lainnya")
X = pd.get_dummies(X, drop_first=True)
print(X.shape, "tingkat batal Resort:", y.mean().round(4))

(40060, 42) tingkat batal Resort: 0.2776


In [19]:
# BAGIAN A (10 mnt): ujian jujur + empat kotak
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=77)   # TODO-1 (solusi)
pemalas = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
print("pemalas akurasi:", round(pemalas.score(Xte, yte), 4))
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                            n_jobs=-1, random_state=77).fit(Xtr, ytr)
tebak = rf.predict(Xte)
dibiarkan, salah_tuduh, lolos, tertangkap = confusion_matrix(yte, tebak).ravel()   # TODO-2 (solusi)
print("dibiarkan", dibiarkan, "salah tuduh", salah_tuduh)
print("lolos", lolos, "tertangkap", tertangkap)
ketepatan = tertangkap / (tertangkap + salah_tuduh)    # TODO-3 (solusi)
jangkauan = tertangkap / (tertangkap + lolos)    # TODO-4 (solusi)
print("akurasi", round(accuracy_score(yte, tebak), 4))
print("ketepatan", round(ketepatan, 4), "jangkauan", round(jangkauan, 4))

pemalas akurasi: 0.7224
dibiarkan 5462 salah tuduh 326
lolos 555 tertangkap 1669
akurasi 0.89
ketepatan 0.8366 jangkauan 0.7504


In [20]:
# BAGIAN B (12 mnt): batas curiga + rupiah
BIAYA_SALAH_TUDUH = 3_000_000     # asumsi resort (boleh diganti)
BIAYA_LOLOS       = 1_500_000     # asumsi resort (boleh diganti)
p = rf.predict_proba(Xte)[:, 1]
baris = []
for batas in np.arange(0.1, 0.91, 0.1):
    curiga = p >= batas                 # TODO-5 (solusi)
    sf = (curiga & (yte == 0)).sum()
    lo = (~curiga & (yte == 1)).sum()
    rupiah = (sf * BIAYA_SALAH_TUDUH + lo * BIAYA_LOLOS) / len(yte) * 1000 / 1e6                 # TODO-6 (solusi)
    baris.append((round(batas, 1), round(precision_score(yte, curiga), 3),
                  round(recall_score(yte, curiga), 3), round(rupiah, 1)))
kolom = ["batas", "ketepatan", "jangkauan", "juta_per_1000"]
tabel = pd.DataFrame(baris, columns=kolom)
print(tabel.to_string(index=False))
murah = tabel["juta_per_1000"].idxmin()
print("termurah:", tabel.loc[murah, "batas"])

 batas  ketepatan  jangkauan  juta_per_1000
   0.1      0.503      0.977          812.9
   0.2      0.610      0.938          525.0
   0.3      0.697      0.885          368.6
   0.4      0.772      0.825          275.4
   0.5      0.837      0.750          226.0
   0.6      0.882      0.661          214.9
   0.7      0.924      0.546          226.3
   0.8      0.953      0.419          259.1
   0.9      0.984      0.281          302.9
termurah: 0.6


In [21]:
# BAGIAN C (8 mnt): uji berputar + siapa yang tertinggal
lipat = StratifiedKFold(n_splits=5, shuffle=True, random_state=77)
model = RandomForestClassifier(n_estimators=100, min_samples_leaf=2, n_jobs=-1, random_state=77)
uji = cross_validate(model, Xtr, ytr, cv=lipat, scoring="recall")    # TODO-7 (solusi)
print("jangkauan tiap lipatan:", uji["test_score"].round(3), "rata-rata", uji["test_score"].mean().round(3))
seg = pd.DataFrame({"segmen": d.loc[Xte.index, "market_segment"].values,
                    "y": yte.values, "t": tebak})
for s, g in seg.groupby("segmen"):                                # TODO-8 (solusi)
    if len(g) > 150:
        print(s, len(g), "jangkauan", round(recall_score(g.y, g.t), 3))

jangkauan tiap lipatan: [0.746 0.739 0.738 0.714 0.739] rata-rata 0.735
Corporate 468 jangkauan 0.443
Direct 1281 jangkauan 0.265
Groups 1175 jangkauan 0.956
Offline TA/TO 1509 jangkauan 0.833
Online TA 3541 jangkauan 0.742
